# 目標の難易度（CEFR レベル）に合わせて語彙を平易化する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/30_language_education/cefr_simplification.ipynb)

NLP クックブック 30-04 ／ 言語教育・リーダビリティ・自動採点

TSAR のような難易度制御付きの語彙平易化を、目標レベル（`B1/B2` 向け vs より易しい `A1/A2` 向け）を切り替えて行いたい。`wordfreq` の Zipf 頻度で難語を特定し、マスク言語モデルで文脈に合う置換候補を生成したうえで、品詞一致・目標レベルの頻度範囲・文埋め込みと NLI による意味保持で候補を選び、文章全体の語彙レベルとリーダビリティの変化を確かめる。

In [1]:
!pip install -q wordfreq textstat sentence-transformers

In [2]:
import re
import nltk
import numpy as np
import pandas as pd
import textstat
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForMaskedLM, AutoModelForSequenceClassification, AutoTokenizer
from wordfreq import zipf_frequency

nltk.download("averaged_perceptron_tagger_eng", quiet=True)

mlm_name = "distilbert-base-uncased"
mlm_tok = AutoTokenizer.from_pretrained(mlm_name)
mlm = AutoModelForMaskedLM.from_pretrained(mlm_name).eval()

sim_model = SentenceTransformer("all-MiniLM-L6-v2")

nli_name = "cross-encoder/nli-deberta-v3-xsmall"
nli_tok = AutoTokenizer.from_pretrained(nli_name)
nli = AutoModelForSequenceClassification.from_pretrained(nli_name).eval()

In [3]:
sentences = [
    "The committee attempted to ameliorate the adverse consequences of the economic downturn.",
    "Smartphones have become ubiquitous in contemporary society.",
    "Researchers must substantiate their hypotheses with empirical evidence.",
    "The new policy will mitigate the financial burden on low-income families.",
    "Careful planning can alleviate severe traffic congestion during peak hours.",
    "The scientist sought to elucidate the complex mechanism behind the phenomenon.",
]

# CEFR 目標レベルごとの頻度範囲（Zipf 尺度: 値が大きいほど高頻度・平易）と意味保持のしきい値
CEFR_TARGETS = {
    "B1/B2": {"trigger_zipf": 4.2, "min_zipf": 4.2, "max_zipf": 5.0, "min_sent_sim": 0.85, "min_nli": 0.65},
    "A1/A2": {"trigger_zipf": 4.6, "min_zipf": 4.9, "max_zipf": 8.0, "min_sent_sim": 0.80, "min_nli": 0.55},
}

def detokenize(tokens):
    return re.sub(r"\s+([.,!?;:])", r"\1", " ".join(tokens))

@torch.no_grad()
def bi_entailment(orig, cands):
    """原文と候補文の双方向の含意確率の最小値（反意語や無関係な語へのすり替わりを防ぐ）"""
    fwd = nli(**nli_tok([orig] * len(cands), cands, padding=True, truncation=True, return_tensors="pt")).logits.softmax(-1)[:, 1]
    bwd = nli(**nli_tok(cands, [orig] * len(cands), padding=True, truncation=True, return_tensors="pt")).logits.softmax(-1)[:, 1]
    return torch.minimum(fwd, bwd).numpy()

@torch.no_grad()
def simplify_to_cefr(sent, cfg):
    tokens = re.findall(r"[A-Za-z]+(?:-[A-Za-z]+)*'?[A-Za-z]*|[.,!?;:]", sent)
    tagged = nltk.pos_tag(tokens)
    orig_emb = sim_model.encode([sent], normalize_embeddings=True, show_progress_bar=False)[0]
    out_tokens = list(tokens)
    edits = []
    for i, (w, tag) in enumerate(tagged):
        if tag[:2] not in ("NN", "VB", "JJ", "RB") or not w.isalpha() or w[0].isupper():
            continue
        z_orig = zipf_frequency(w, "en")
        if z_orig >= cfg["trigger_zipf"]:
            continue
        cur_sent = detokenize(out_tokens)
        masked_sent = detokenize(out_tokens[:i] + [mlm_tok.mask_token] + out_tokens[i + 1:])
        enc = mlm_tok(cur_sent, masked_sent, return_tensors="pt")
        pos = (enc["input_ids"][0] == mlm_tok.mask_token_id).nonzero()[0, 0]
        top = mlm(**enc).logits[0, pos].softmax(-1).topk(80)

        used = {t.lower() for j, t in enumerate(out_tokens) if j != i and t.isalpha()}
        cands = []
        for idx, p in zip(top.indices.tolist(), top.values.tolist()):
            cw = mlm_tok.convert_ids_to_tokens(idx)
            if not cw.isalpha() or len(cw) < 2 or cw == w.lower() or cw in used or p < 1e-4:
                continue
            zc = zipf_frequency(cw, "en")
            if not (cfg["min_zipf"] <= zc <= cfg["max_zipf"] and zc > z_orig + 0.3):
                continue
            cand_tokens = out_tokens[:i] + [cw] + out_tokens[i + 1:]
            if nltk.pos_tag(cand_tokens)[i][1] != tag:
                continue
            cands.append((cw, zc, p, detokenize(cand_tokens)))
        if not cands:
            continue
        cs_embs = sim_model.encode([c[3] for c in cands], normalize_embeddings=True, show_progress_bar=False)
        s_sims = cs_embs @ orig_emb
        shortlist = sorted(
            [(cw, zc, p, cs, float(ss)) for (cw, zc, p, cs), ss in zip(cands, s_sims) if ss >= cfg["min_sent_sim"]],
            key=lambda x: -(0.7 * x[4] + 0.3 * min(x[2] * 10, 1.0)),
        )[:10]
        if not shortlist:
            continue
        nli_scores = bi_entailment(cur_sent, [x[3] for x in shortlist])
        valid = [(cw, zc, p, ss, float(ns)) for (cw, zc, p, _, ss), ns in zip(shortlist, nli_scores) if ns >= cfg["min_nli"]]
        if valid:
            best = max(valid, key=lambda x: 0.5 * x[4] + 0.4 * x[3] + 0.1 * min(x[2] * 10, 1.0))
            out_tokens[i] = best[0]
            edits.append((w, round(z_orig, 2), best[0], round(best[1], 2), round(best[4], 2)))
    return detokenize(out_tokens), edits

In [4]:
def corpus_metrics(texts):
    words = [w for t in texts for w in re.findall(r"[A-Za-z]+", t) if zipf_frequency(w, "en") > 0]
    return {
        "平均 Zipf 頻度": float(np.mean([zipf_frequency(w, "en") for w in words])),
        "低頻度語 (<4.5) 割合": float(np.mean([zipf_frequency(w, "en") < 4.5 for w in words])),
        "Flesch-Kincaid Grade": float(np.mean([textstat.flesch_kincaid_grade(t) for t in texts])),
    }

results = {"原文": sentences}
for level, cfg in CEFR_TARGETS.items():
    outs = []
    print(f"=== {level} ===")
    for s in sentences:
        out, edits = simplify_to_cefr(s, cfg)
        outs.append(out)
        print(f"  {out}")
        if edits:
            print("    置換:", ", ".join(f"{o}({zo})→{c}({zc}, NLI={ns})" for o, zo, c, zc, ns in edits))
    results[level] = outs

summary = pd.DataFrame({k: corpus_metrics(v) for k, v in results.items()}).T.round(2)
print("\n=== レベル別の語彙頻度とリーダビリティの変化 ===")
print(summary.to_string())

=== B1/B2 ===
  The committee attempted to address the negative consequences of the economic downturn.
    置換: ameliorate(2.56)→address(4.91, NLI=0.98), adverse(3.89)→negative(4.79, NLI=0.99)
  Smartphones have become everyday in contemporary society.
    置換: ubiquitous(3.42)→everyday(4.47, NLI=0.98)
  Researchers must confirm their theories with empirical evidence.
    置換: substantiate(2.83)→confirm(4.4, NLI=0.99), hypotheses(3.2)→theories(4.22, NLI=0.99)
  The new policy will reduce the financial burden on low-income families.
    置換: mitigate(3.54)→reduce(4.73, NLI=0.99)
  Careful planning can reduce severe traffic trouble during peak hours.
    置換: alleviate(3.52)→reduce(4.73, NLI=0.99), congestion(3.66)→trouble(4.79, NLI=0.96)
  The scientist sought to explain the complex mechanism behind the concept.
    置換: elucidate(2.7)→explain(4.89, NLI=0.99), phenomenon(4.13)→concept(4.77, NLI=0.98)
=== A1/A2 ===
  The committee tried to address the bad results of the economic downturn.
    

- 難語のしきい値 (`trigger_zipf`) と置換候補の頻度範囲 (`min_zipf`〜`max_zipf`) を目標 CEFR レベルに合わせて変えることで、`B1/B2` では *address / negative / confirm / theories / reduce* のように中級語彙へ、`A1/A2` では *tried / bad / results / common / support / lower* のように基礎語彙へ段階的に平易化され、平均 Zipf 頻度と Flesch-Kincaid Grade が単調に易しくなる。
- マスク言語モデルの確率と文埋め込みの類似度だけでは、同じ文脈に収まる反意語（例: *mitigate* → *increase*）を通すことがある。原文と候補文の双方向の NLI 含意確率を組み合わせることで意味の反転を弾ける。
- 語彙の置換だけでは文構造や連語（コロケーション）の不自然さ（例: *severe traffic* の後ろの *peak hours* → *high hours*）が残ることがあり、単語単位の言い換えの限界も見える。文全体の書き換えや日本語の平易化については、要約と平易化の章の「難しい単語を見つけてやさしい単語に言い換える（語彙平易化）」「文章をやさしい日本語に書き換える（テキスト平易化）」も参照。